# Sesión 3 — Desbalance de Clases y Series de Tiempo

Código de los conceptos de la sesión y el ejercicio para practicarlos.

## Retomamos: dataset de fraude de la Sesión 2

Esta sesión abre con **desbalance de clases**, sobre el mismo dataset de
fraude submuestreado de la Sesión 2. Recargamos rápidamente los datos y un
Random Forest de referencia para que este notebook corra de forma
independiente.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Mismo dataset de fraude submuestreado de la Sesión 2
d_fraude = fetch_openml("creditcard", version=1, as_frame=True, parser="auto")
df_full_fraude = d_fraude.frame
df_full_fraude["Class"] = df_full_fraude["Class"].astype(int)

fraude = df_full_fraude[df_full_fraude["Class"] == 1]
no_fraude = df_full_fraude[df_full_fraude["Class"] == 0].sample(n=20_000, random_state=RANDOM_STATE)
df_fraude = pd.concat([fraude, no_fraude], axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

X_fraude = df_fraude.drop(columns=["Class"])
y_fraude = df_fraude["Class"]
X_train_fraude, X_test_fraude, y_train_fraude, y_test_fraude = train_test_split(
    X_fraude, y_fraude, test_size=0.2, stratify=y_fraude, random_state=RANDOM_STATE
)

rf_fraude = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
rf_fraude.fit(X_train_fraude, y_train_fraude)
y_pred_rf = rf_fraude.predict(X_test_fraude)


def resumen_metricas(nombre, y_true, y_pred, y_score=None):
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_score) if y_score is not None else np.nan,
    }

print("Retomamos el dataset de fraude submuestreado:", df_fraude.shape, "· Random Forest re-entrenado.")

## 1. Desbalance de clases

### 4.1 Motivación: un caso real

Un caso mencionado en clase: una empresa de energía eléctrica quiere predecir
la **probabilidad de falla** en cada zona/hora, para asignar de forma
preventiva las brigadas de reparación disponibles. En este tipo de problema,
la proporción de casos positivos (falla real) suele ser de apenas **~1.8%**
de las observaciones: la inmensa mayoría de las combinaciones zona-hora no
presentan falla.

Nuestro dataset de fraude submuestreado tiene una situación análoga: ~2.4%
de casos positivos. Vamos a ver por qué esto rompe la intuición habitual de
usar **accuracy** como métrica de desempeño.

### 4.2 Por qué accuracy engaña

Un modelo que **predice "0" siempre**, sin mirar ni una sola variable, ya
tiene una accuracy muy alta simplemente porque la clase mayoritaria domina
el dataset. Verifiquémoslo:

In [ ]:
dummy = DummyClassifier(strategy="constant", constant=0)
dummy.fit(X_train_fraude, y_train_fraude)
y_pred_dummy = dummy.predict(X_test_fraude)

print(f"Accuracy del clasificador 'todo es 0': {accuracy_score(y_test_fraude, y_pred_dummy):.4f}")
print(classification_report(y_test_fraude, y_pred_dummy, digits=3, target_names=["normal", "fraude"], zero_division=0))

El clasificador "todo es 0" tiene ~97.6% de accuracy (igual al porcentaje de
la clase mayoritaria) pero **recall = 0** sobre la clase de interés
(fraude): nunca detecta ni un solo fraude. En un caso real de brigadas de
reparación, este modelo sería inútil (nunca despacharía una brigada), a
pesar de su accuracy aparentemente excelente.

Por eso, en problemas desbalanceados usamos métricas que sí reflejan el
desempeño sobre la clase minoritaria:

- **Precision**: de las veces que predije positivo, ¿cuántas eran realmente
  positivas?
- **Recall**: de todos los positivos reales, ¿cuántos detecté?
- **F1**: media armónica de precision y recall.
- **PR-AUC** (área bajo la curva Precision-Recall): resume el trade-off
  precision/recall en todos los umbrales de decisión posibles; es más
  informativa que el AUC-ROC tradicional cuando la clase positiva es rara.

### 4.3 Técnicas de balanceo

- **Random Undersampling**: elimina aleatoriamente muestras de la clase
  mayoritaria hasta emparejar (o acercar) su tamaño al de la clase
  minoritaria. Ventaja: rápido y simple. Desventaja: se descarta información
  potencialmente útil de la clase mayoritaria.
- **Random Oversampling**: duplica aleatoriamente muestras de la clase
  minoritaria (con reemplazo) hasta emparejar su tamaño al de la mayoritaria.
  Ventaja: no se pierde información de la clase mayoritaria. Desventaja:
  puede favorecer overfitting sobre las mismas observaciones minoritarias
  repetidas.
- **Tomek Links**: identifica pares de observaciones de clases opuestas que
  son **mutuamente el vecino más cercano** entre sí (es decir, ninguna otra
  observación está más cerca de ninguna de las dos). Estos pares suelen
  estar en la frontera de decisión o representar ruido; se elimina la
  observación de la clase **mayoritaria** de cada par, "limpiando" la
  frontera entre clases (a diferencia del undersampling puro al azar, que no
  mira dónde están las observaciones).
- **NearMiss**: variante de undersampling *informado*, que en vez de
  eliminar al azar conserva las observaciones de la clase mayoritaria más
  cercanas (o con ciertos criterios de distancia) a la clase minoritaria,
  buscando mantener los casos "difíciles"/informativos de la frontera de
  decisión.

Implementamos versiones simples de undersampling y oversampling con pandas
(no usamos la librería `imbalanced-learn`, que no está instalada en el
proyecto, para evitar riesgos de instalación en las máquinas de los
estudiantes).

### 4.4 Demo: undersampling manual

In [ ]:
train_bal = X_train_fraude.copy()
train_bal["Class"] = y_train_fraude.values

mayoritaria = train_bal[train_bal["Class"] == 0]
minoritaria = train_bal[train_bal["Class"] == 1]

print(f"Antes  -> mayoritaria: {len(mayoritaria)}, minoritaria: {len(minoritaria)}")

mayoritaria_under = mayoritaria.sample(n=len(minoritaria), random_state=RANDOM_STATE)
train_under = pd.concat([mayoritaria_under, minoritaria]).sample(frac=1.0, random_state=RANDOM_STATE)

print(f"Después -> mayoritaria: {len(mayoritaria_under)}, minoritaria: {len(minoritaria)}")

X_train_under = train_under.drop(columns=["Class"])
y_train_under = train_under["Class"]

In [ ]:
rf_under = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
rf_under.fit(X_train_under, y_train_under)
y_pred_rf_under = rf_under.predict(X_test_fraude)

print(classification_report(y_test_fraude, y_pred_rf_under, digits=3, target_names=["normal", "fraude"]))

### 4.5 Demo: oversampling manual

In [ ]:
minoritaria_over = minoritaria.sample(n=len(mayoritaria), replace=True, random_state=RANDOM_STATE)
train_over = pd.concat([mayoritaria, minoritaria_over]).sample(frac=1.0, random_state=RANDOM_STATE)

print(f"Después de oversampling -> mayoritaria: {len(mayoritaria)}, minoritaria (con repetición): {len(minoritaria_over)}")

X_train_over = train_over.drop(columns=["Class"])
y_train_over = train_over["Class"]

rf_over = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
rf_over.fit(X_train_over, y_train_over)
y_pred_rf_over = rf_over.predict(X_test_fraude)

print(classification_report(y_test_fraude, y_pred_rf_over, digits=3, target_names=["normal", "fraude"]))

### 4.6 Comparación antes / después del balanceo

In [ ]:
comparacion_balanceo = pd.DataFrame([
    resumen_metricas("RF (datos originales, desbalanceados)", y_test_fraude, y_pred_rf, rf_fraude.predict_proba(X_test_fraude)[:, 1]),
    resumen_metricas("RF + undersampling", y_test_fraude, y_pred_rf_under, rf_under.predict_proba(X_test_fraude)[:, 1]),
    resumen_metricas("RF + oversampling", y_test_fraude, y_pred_rf_over, rf_over.predict_proba(X_test_fraude)[:, 1]),
]).set_index("modelo")

comparacion_balanceo.round(3)

Observa que el accuracy apenas cambia entre las tres variantes (siempre va a
ser alto porque la clase mayoritaria domina el test set), pero **recall**,
**F1** y **PR-AUC** sí cambian de forma más notoria: el balanceo típicamente
mejora el recall (detectamos más fraudes reales) aunque puede sacrificar algo
de precision (más falsas alarmas). Cuál balance es preferible depende del
costo de negocio de cada tipo de error (un fraude no detectado vs. una
transacción legítima marcada como sospechosa).

---

---

Cerramos desbalance de clases. Cambiamos ahora de problema por completo:
**series de tiempo** — otro escenario donde la validación "estándar"
(esta vez, K-Fold aleatorio) también falla, aunque por una razón distinta
a la de hoy (aquí es por *fuga de información temporal*, no por
desbalance).

## 2. ¿Qué es una serie de tiempo?

Una **serie de tiempo** es un conjunto de observaciones registradas
**secuencialmente en el tiempo**, típicamente a intervalos regulares (cada
hora, cada día, cada mes, etc.).

Notación estándar:

$$Y_t = \{y_1, y_2, \dots, y_T\}$$

donde el subíndice $t$ indica el **orden temporal** de la observación (no es
un identificador arbitrario como en una fila de una tabla cualquiera: el
orden importa y no se puede alterar).

### Aplicaciones típicas

- **Demanda / ventas**: unidades vendidas por día, pronóstico de demanda para
  planeación de inventarios.
- **Finanzas**: precio de una acción, tasa de cambio, retornos diarios.
- **Energía**: demanda eléctrica horaria, generación de una planta solar.
- **Operaciones**: número de tickets de soporte por hora, latencia de un
  sistema, número de llamadas a un call center.

En todos estos casos el objetivo típico es **pronosticar** (forecast) valores
futuros a partir del historial observado.

## 3. ¿Por qué una serie de tiempo es distinta a un problema supervisado clásico?

En un problema de regresión/clasificación "clásico" (por ejemplo, predecir el
precio de una casa a partir de sus características) asumimos que las
observaciones son **independientes e idénticamente distribuidas (i.i.d.)**.
En series de tiempo esa suposición **se rompe**:

1. **Las observaciones NO son independientes entre sí.** El valor de hoy está
   correlacionado con el valor de ayer, de hace una semana, de hace un año
   (esto se llama **autocorrelación**). Ignorar esta dependencia bota
   información valiosa.
2. **No se puede hacer *shuffle* (mezcla aleatoria) de los datos.** El orden
   temporal es información en sí mismo. Si mezclamos las filas, destruimos la
   estructura que justamente queremos modelar — y, como veremos más adelante,
   esto también rompe la validación cruzada tradicional.
3. **La media y la varianza pueden cambiar con el tiempo.** Una serie puede
   tener tendencia (la media sube o baja) o cambios de varianza (la serie se
   vuelve más o menos volátil). Esto viola el supuesto de estacionariedad que
   muchos modelos clásicos de series de tiempo necesitan (no así los modelos
   de ML, que son más flexibles al respecto, aunque igual conviene entenderlo).

Estas diferencias son la razón por la que dedicamos una sesión completa a
series de tiempo en lugar de tratarlas como "una tabla más".

## Configuración del entorno

Cargamos las librerías que usaremos en toda la sesión. Todo lo que hacemos
aquí se resuelve con `numpy`, `pandas`, `matplotlib` y `scikit-learn`
(no usamos `statsmodels`, `prophet` ni librerías de deep learning).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paleta de colores fija que usaremos en toda la sesión (consistencia visual):
# un color por "rol" semántico, no por gráfico.
COLOR_SERIE = "#4C72B0"      # serie observada / real
COLOR_TENDENCIA = "#DD8452"  # tendencia
COLOR_ESTACIONAL = "#55A868" # componente estacional
COLOR_RESIDUO = "#8C8C8C"    # residuo / ruido
COLOR_NAIVE = "#8C8C8C"      # baseline naive
COLOR_LINEAL = "#C44E52"     # regresión lineal
COLOR_HGB = "#4C72B0"        # HistGradientBoosting

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## Dataset conceptual: AirPassengers

Serie mensual clásica: número de pasajeros de aerolíneas internacionales
(en miles), de enero de 1949 a diciembre de 1960. Es *el* ejemplo canónico
para introducir tendencia + estacionalidad porque ambas son muy visibles.

In [ ]:
url_air = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/AirPassengers.csv"
air = pd.read_csv(url_air)

# La columna `time` viene como año decimal (1949.000, 1949.083, ...).
# La convertimos a una fecha real (inicio de cada mes) para trabajar con pandas.
air = air.sort_values("time").reset_index(drop=True)
air["date"] = pd.date_range(start="1949-01-01", periods=len(air), freq="MS")
air = air.rename(columns={"value": "passengers"})[["date", "passengers"]]

print(air.shape)
air.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(air["date"], air["passengers"], color=COLOR_SERIE, linewidth=1.5)
ax.set_title("AirPassengers: pasajeros aéreos internacionales, 1949–1960")
ax.set_xlabel("Fecha")
ax.set_ylabel("Pasajeros (miles)")
fig.tight_layout()
plt.show()

Se ven claramente dos cosas a simple vista:

- Una **tendencia creciente** de largo plazo (más gente vuela con el paso de
  los años).
- Un **patrón estacional** que se repite cada 12 meses (picos en la temporada
  alta de vacaciones).
- La **amplitud** de las oscilaciones estacionales **crece** a medida que la
  serie crece — esto es una pista de que la serie tiene un comportamiento
  **multiplicativo**, no aditivo (lo confirmamos abajo).

## 4. Componentes de una serie de tiempo

Toda serie de tiempo se puede pensar como la combinación de tres componentes:

- **Tendencia ($T_t$)**: el movimiento de largo plazo (sube, baja, o se
  mantiene estable a lo largo de muchos periodos).
- **Estacionalidad ($S_t$)**: un patrón que se repite a intervalos **fijos y
  conocidos** (cada 12 meses, cada 7 días, cada 24 horas...).
- **Residuo / ruido ($\varepsilon_t$)**: lo que queda después de remover
  tendencia y estacionalidad; en teoría debería verse como ruido aleatorio sin
  estructura.

### Descomposición aditiva vs. multiplicativa

**Aditiva** — la amplitud de la estacionalidad se mantiene **constante** sin
importar el nivel de la serie:

$$Y_t = T_t + S_t + \varepsilon_t$$

**Multiplicativa** — la amplitud de la estacionalidad **crece o decrece**
proporcionalmente al nivel de la tendencia (como en AirPassengers: entre más
alto el nivel general, más grandes son los picos y valles estacionales):

$$Y_t = T_t \times S_t \times \varepsilon_t$$

Una forma práctica de decidir cuál usar: graficar la serie y ver si la
amplitud de las oscilaciones es aproximadamente constante (aditiva) o si
crece con el nivel (multiplicativa). También se puede aplicar logaritmo a una
serie multiplicativa para volverla aditiva, porque
$\log(T_t \cdot S_t \cdot \varepsilon_t) = \log T_t + \log S_t + \log \varepsilon_t$.

`statsmodels` trae una función `seasonal_decompose` que hace esto
automáticamente, pero como no la tenemos instalada, la implementamos
manualmente — es un ejercicio útil porque muestra exactamente qué hay "detrás"
de la descomposición:

1. **Tendencia**: media móvil centrada con ventana = periodo estacional (12
   para datos mensuales).
2. **Estacionalidad**: para cada mes del año, promediamos el residuo
   `serie - tendencia` (aditivo) o `serie / tendencia` (multiplicativo) sobre
   todos los años disponibles.
3. **Residuo**: lo que queda después de remover tendencia y estacionalidad.

In [ ]:
def descomponer(serie: pd.Series, periodo: int, tipo: str = "additive") -> pd.DataFrame:
    # Descomposición simple (tendencia + estacionalidad + residuo).
    # Implementación manual equivalente en espíritu a
    # `statsmodels.tsa.seasonal.seasonal_decompose`, pero sin esa dependencia.
    assert tipo in {"additive", "multiplicative"}

    # 1) Tendencia: media móvil centrada de ventana = periodo.
    tendencia = serie.rolling(window=periodo, center=True).mean()

    # 2) Detrend: quitamos la tendencia para aislar la estacionalidad.
    if tipo == "additive":
        detrend = serie - tendencia
    else:
        detrend = serie / tendencia

    # 3) Estacionalidad: promedio del detrend agrupado por posición dentro
    #    del ciclo estacional (ej. mismo mes en distintos años).
    posicion = np.arange(len(serie)) % periodo
    promedio_por_posicion = pd.Series(detrend.values, index=posicion).groupby(level=0).mean()
    estacional = pd.Series(promedio_por_posicion[posicion].values, index=serie.index)

    # 4) Residuo: lo que sobra.
    if tipo == "additive":
        residuo = serie - tendencia - estacional
    else:
        residuo = serie / (tendencia * estacional)

    return pd.DataFrame(
        {"observado": serie, "tendencia": tendencia, "estacional": estacional, "residuo": residuo}
    )


desc_mult = descomponer(air.set_index("date")["passengers"], periodo=12, tipo="multiplicative")

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
axes[0].plot(desc_mult.index, desc_mult["observado"], color=COLOR_SERIE)
axes[0].set_title("Observado")
axes[1].plot(desc_mult.index, desc_mult["tendencia"], color=COLOR_TENDENCIA)
axes[1].set_title("Tendencia (media móvil centrada, ventana=12)")
axes[2].plot(desc_mult.index, desc_mult["estacional"], color=COLOR_ESTACIONAL)
axes[2].set_title("Estacionalidad (multiplicativa)")
axes[3].plot(desc_mult.index, desc_mult["residuo"], color=COLOR_RESIDUO)
axes[3].set_title("Residuo")
fig.suptitle("Descomposición multiplicativa de AirPassengers", y=1.02)
fig.tight_layout()
plt.show()

El residuo multiplicativo queda oscilando alrededor de 1.0 y sin un
patrón obvio, lo que confirma que el modelo multiplicativo describe bien esta
serie (si hubiéramos usado descomposición aditiva, el residuo mostraría
todavía un patrón creciente de varianza — puedes comprobarlo cambiando
`tipo="additive"` arriba).

## 5. Estacionariedad

Una serie es (débilmente) **estacionaria** si:

1. Su **media** es constante en el tiempo (no tiene tendencia).
2. Su **varianza** es constante en el tiempo (no se vuelve más o menos
   volátil).
3. Su **autocovarianza** entre $y_t$ y $y_{t+k}$ depende solamente del rezago
   $k$, no del momento $t$ en el que se mide.

Muchos modelos clásicos de series de tiempo (como ARIMA) **necesitan** que la
serie sea estacionaria para funcionar bien. Los modelos de ML que usaremos en
esta sesión son más flexibles al respecto, pero igual es un concepto clave
para entender el comportamiento de una serie.

### ¿Cómo detectar si una serie es estacionaria?

- **Inspección visual**: ¿la serie tiene tendencia o cambia de varianza a
  simple vista?
- **Estadísticos móviles (rolling mean / rolling std)**: si la media y la
  desviación estándar calculadas sobre una ventana móvil cambian de forma
  sistemática en el tiempo, la serie **no** es estacionaria.
- **Test de Dickey-Fuller aumentado (ADF)**: un test de hipótesis formal.
  - $H_0$: la serie tiene una **raíz unitaria** (no es estacionaria).
  - $H_1$: la serie es estacionaria.
  - Si el p-value es menor a un umbral (ej. 0.05), rechazamos $H_0$ y
    concluimos que la serie es estacionaria.
  - En `statsmodels` se ejecutaría con `statsmodels.tsa.stattools.adfuller`.
    Como no tenemos esa librería instalada, **no lo ejecutamos** en este
    notebook, pero es importante que sepas que existe y cómo se interpreta:
    es el complemento formal a la inspección visual de rolling stats que sí
    hacemos abajo.

Veamos las estadísticas móviles de AirPassengers (claramente no estacionaria):

In [ ]:
ventana = 12
rolling_mean = air.set_index("date")["passengers"].rolling(ventana).mean()
rolling_std = air.set_index("date")["passengers"].rolling(ventana).std()

fig, ax = plt.subplots()
ax.plot(air["date"], air["passengers"], color=COLOR_SERIE, alpha=0.4, label="Serie original")
ax.plot(rolling_mean.index, rolling_mean, color=COLOR_TENDENCIA, linewidth=2, label=f"Media móvil ({ventana}m)")
ax.plot(rolling_std.index, rolling_std, color=COLOR_ESTACIONAL, linewidth=2, label=f"Desv. estándar móvil ({ventana}m)")
ax.set_title("AirPassengers: media y desviación móvil crecientes → NO estacionaria")
ax.legend()
fig.tight_layout()
plt.show()

La media móvil crece claramente (tendencia) y la desviación estándar
móvil también crece (varianza no constante). Ambas señales visuales indican
que la serie **no es estacionaria**.

### Cómo volver una serie estacionaria

- **Diferenciación** (la más común): $Y'_t = Y_t - Y_{t-1}$. Remueve la
  tendencia.
- **Diferenciación estacional**: $Y'_t = Y_t - Y_{t-m}$ (con $m$ = periodo
  estacional, ej. 12 para datos mensuales). Remueve la estacionalidad.
- **Transformación logarítmica**: $Y'_t = \log(Y_t)$. Estabiliza series cuya
  varianza crece con el nivel (típico en series multiplicativas).
- **Transformación Box-Cox**: una familia más general que incluye el log
  como caso particular, parametrizada por $\lambda$:
  $Y'_t = \dfrac{Y_t^{\lambda} - 1}{\lambda}$ si $\lambda \neq 0$, y
  $\log(Y_t)$ si $\lambda = 0$. Busca automáticamente la transformación de
  potencia que más estabiliza la varianza.

Veamos el efecto de la diferenciación simple sobre AirPassengers:

In [ ]:
passengers = air.set_index("date")["passengers"]
diff_1 = passengers.diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(passengers.index, passengers, color=COLOR_SERIE)
axes[0].set_title("Serie original (no estacionaria)")
axes[1].plot(diff_1.index, diff_1, color=COLOR_TENDENCIA)
axes[1].axhline(0, color=COLOR_RESIDUO, linewidth=1, linestyle="--")
axes[1].set_title("Primera diferencia $Y_t - Y_{t-1}$ (tendencia removida)")
fig.tight_layout()
plt.show()

print("Media móvil (12m) de la serie original — inicio vs. fin:")
print(f"  primeros 12 meses: {passengers.rolling(12).mean().dropna().iloc[0]:.1f}")
print(f"  últimos 12 meses:  {passengers.rolling(12).mean().dropna().iloc[-1]:.1f}")
print()
print("Media móvil (12m) de la serie diferenciada — inicio vs. fin:")
diff_roll = diff_1.rolling(12).mean().dropna()
print(f"  primeros 12 meses: {diff_roll.iloc[0]:.2f}")
print(f"  últimos 12 meses:  {diff_roll.iloc[-1]:.2f}")

La diferenciación remueve la tendencia (la media móvil ya no crece
sistemáticamente), pero nota que la varianza todavía crece con el tiempo — eso
es evidencia de que esta serie es multiplicativa y necesitaría además una
transformación logarítmica antes de diferenciar para quedar completamente
estacionaria. Te lo dejamos como ejercicio mental (o para el ejercicio de
práctica al final).

## 6. ARIMA(p, d, q) — panorama conceptual

**ARIMA** (AutoRegressive Integrated Moving Average) es la familia de modelos
clásicos más usada en series de tiempo univariadas. Tiene tres componentes,
cada uno con su propio orden:

- **AR(p) — AutoRegresivo**: el valor actual depende linealmente de sus **p**
  valores pasados.
  $$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \dots + \phi_p y_{t-p} + \varepsilon_t$$
- **I(d) — Integrado**: en lugar de modelar $y_t$ directamente, se modela la
  serie **diferenciada d veces** hasta volverla estacionaria (ver sección 4).
- **MA(q) — Media Móvil**: el valor actual depende linealmente de los **q**
  errores (residuos) pasados de pronóstico, no de los valores pasados.
  $$y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \dots + \theta_q \varepsilon_{t-q}$$

Combinando los tres: ARIMA(p, d, q) modela la serie diferenciada d veces como
una combinación de sus propios rezagos (AR) y de los errores pasados (MA). Los
órdenes p, d, q usualmente se eligen inspeccionando las funciones de
autocorrelación (ACF) y autocorrelación parcial (PACF), o por búsqueda
automática (ej. `auto_arima`).

**En esta sesión no implementamos ARIMA** porque requiere `statsmodels` (no
instalado en el entorno del curso). El foco práctico de esta sesión es una
alternativa muy usada en la industria — especialmente en pronóstico de
demanda con muchas series en paralelo: **reformular el problema de series de
tiempo como un problema de aprendizaje supervisado** y aplicarle los modelos
de ML que ya conocemos del curso. Vamos a esa parte.

## 7. Reformular una serie de tiempo como problema supervisado

La idea central: en lugar de modelar $y_t$ con un modelo especializado de
series de tiempo (como ARIMA), construimos una **matriz de features $X$** y
un **vector objetivo $y$** donde cada fila representa un instante $t$, y las
columnas (features) son información **disponible antes de $t$**:

$$\hat{y}_t = f(y_{t-1}, y_{t-2}, \dots, y_{t-k},\ \text{exógenas}_t,\ \text{calendario}_t)$$

Una vez reformulado así, **cualquier modelo de ML supervisado** (regresión
lineal, árboles, gradient boosting...) se puede aplicar directamente. Este es
el enfoque dominante en problemas de pronóstico de demanda a gran escala
(muchas series/SKUs en paralelo), porque permite entrenar un solo modelo
sobre todas las series a la vez y aprovechar variables exógenas fácilmente.

### Tipos de features

**1. Variables de rezago (lags)**

El valor de la serie hace $k$ periodos: $y_{t-k}$. Capturan directamente la
dependencia temporal / autocorrelación. La elección de qué rezagos incluir
depende del dominio: por ejemplo, en datos horarios tiene sentido incluir
$y_{t-1}$ (hace 1 hora), $y_{t-24}$ (mismo momento ayer) y $y_{t-168}$ (mismo
momento la semana pasada).

**2. Variables de calendario**

- **Categóricas**: día de la semana, mes, trimestre, hora del día.
- **Booleanas**: fin de semana, día festivo, si hubo una promoción activa.
- **Tiempo absoluto**: año, número de día desde el inicio de la serie (captura
  tendencia de largo plazo).
- **Cíclicas (seno/coseno)** — **muy importante**: variables como "hora del
  día" o "mes" son categóricas pero tienen una estructura **circular**: la
  hora 23 está tan cerca de la hora 0 como la hora 22 lo está de la hora 0...
  pero si codificamos la hora como un entero simple (0, 1, 2, ..., 23), un
  modelo lineal (o cualquier modelo que use distancias/valores ordinales) ve
  la hora 23 y la hora 0 como **muy lejanas**, cuando en realidad son
  consecutivas. La solución es la **codificación cíclica**:

  $$\sin\left(\frac{2\pi \cdot \text{hora}}{24}\right), \qquad
    \cos\left(\frac{2\pi \cdot \text{hora}}{24}\right)$$

  Esto mapea cada hora a un punto en un círculo unitario, preservando la
  noción de que la hora 23 y la hora 0 están cerca. Se aplica igual para día
  de la semana (periodo 7), mes (periodo 12), día del año (periodo ~365), etc.

**3. Variables rolling (ventana móvil)**

Estadísticos calculados sobre una ventana de tiempo pasado: media móvil,
desviación estándar móvil, mínimo/máximo móvil. Por ejemplo, la media móvil
de las últimas 24 horas resume el "nivel reciente" de la serie de forma más
estable que un solo lag.

**Precaución importante**: todas estas variables deben calcularse usando
**solamente información pasada** respecto al instante que queremos predecir —
de lo contrario hay fuga de información (*leakage*) y el modelo parecerá
funcionar mucho mejor de lo que realmente funcionaría en producción.

Vamos a construir todo esto con el dataset de **Bike Sharing** (conteo
horario de alquiler de bicicletas, con variables de clima y calendario ya
incluidas — un problema de "demanda" muy similar en espíritu al de ventas o
energía).

### Cargar el dataset Bike Sharing (UCI)

Lo descargamos en tiempo de ejecución (no se commitea al repo).

In [ ]:
# Datos alojados en el almacenamiento del curso.
CDN = "https://d3qixogk4zgixq.cloudfront.net/data/sesiones/sesion_04_bike_sharing"
DATA_DIR = Path("bike_sharing_data")
DATA_DIR.mkdir(exist_ok=True)

for archivo in ("hour.csv", "day.csv"):
    destino = DATA_DIR / archivo
    if not destino.exists():
        resp = requests.get(f"{CDN}/{archivo}", timeout=60)
        resp.raise_for_status()
        destino.write_bytes(resp.content)

bike = pd.read_csv(DATA_DIR / "hour.csv")
print(bike.shape)
bike.head()

In [ ]:
# Construimos un datetime real combinando la fecha (`dteday`) y la hora (`hr`).
bike["datetime"] = pd.to_datetime(bike["dteday"]) + pd.to_timedelta(bike["hr"], unit="h")
bike = bike.sort_values("datetime").reset_index(drop=True)

print(f"Rango: {bike['datetime'].min()} → {bike['datetime'].max()}")
print(f"Filas: {len(bike)}")
bike[["datetime", "hr", "weekday", "mnth", "holiday", "workingday", "temp", "hum", "cnt"]].head()

Columnas relevantes del dataset original (ya vienen con el UCI Bike
Sharing Dataset):

- `cnt`: número total de bicicletas alquiladas en esa hora — **nuestra
  variable objetivo**.
- `hr`, `weekday`, `mnth`: hora (0-23), día de la semana (0-6) y mes (1-12).
- `holiday`: 1 si es día festivo.
- `workingday`: 1 si es día laboral (no fin de semana ni festivo).
- `temp`, `atemp`, `hum`, `windspeed`: variables climáticas (exógenas),
  normalizadas por los autores del dataset.

Vamos a construir nuestras propias variables de lags, calendario (incluyendo
codificación cíclica) y rolling **encima** de estas columnas.

In [ ]:
fig, ax = plt.subplots()
muestra = bike.iloc[: 24 * 14]  # primeras dos semanas
ax.plot(muestra["datetime"], muestra["cnt"], color=COLOR_SERIE, linewidth=1)
ax.set_title("Bike Sharing: conteo horario de alquileres (primeras 2 semanas)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Alquileres por hora")
fig.tight_layout()
plt.show()

Se ve un patrón diario muy marcado (picos en horas pico de
entrada/salida laboral) y diferencias entre días de semana y fines de semana
— justo el tipo de estructura que las variables de calendario y los lags
deben capturar.

### Ingeniería de features: lags, calendario y rolling

Construimos:

- **Lags**: `cnt` hace 1 hora, hace 24 horas (mismo momento ayer) y hace 168
  horas (mismo momento hace una semana).
- **Calendario cíclico**: seno/coseno de hora (periodo 24), día de la semana
  (periodo 7) y mes (periodo 12).
- **Calendario simple**: `is_weekend`, `holiday` (ya viene en el dataset).
- **Rolling**: media móvil y desviación estándar móvil de las últimas 24
  horas, calculadas usando `.shift(1)` antes del rolling para no incluir la
  hora actual (evitar leakage).

In [ ]:
df = bike.copy()

# --- Lags ---
df["lag_1"] = df["cnt"].shift(1)
df["lag_24"] = df["cnt"].shift(24)
df["lag_168"] = df["cnt"].shift(168)

# --- Rolling (usamos shift(1) antes del rolling para solo usar info pasada) ---
df["rolling_mean_24"] = df["cnt"].shift(1).rolling(window=24).mean()
df["rolling_std_24"] = df["cnt"].shift(1).rolling(window=24).std()

# --- Calendario simple ---
df["is_weekend"] = (df["weekday"].isin([0, 6])).astype(int)  # 0=domingo, 6=sábado en este dataset
df["holiday"] = df["holiday"].astype(int)

# --- Calendario cíclico (seno/coseno) ---
df["hour_sin"] = np.sin(2 * np.pi * df["hr"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hr"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)
df["month_sin"] = np.sin(2 * np.pi * (df["mnth"] - 1) / 12)
df["month_cos"] = np.cos(2 * np.pi * (df["mnth"] - 1) / 12)

# Variables exógenas de clima que ya vienen en el dataset.
df["temp_exog"] = df["temp"]
df["hum_exog"] = df["hum"]
df["windspeed_exog"] = df["windspeed"]

# Quitamos las filas iniciales sin historia suficiente para lag_168.
df_model = df.dropna(subset=["lag_1", "lag_24", "lag_168", "rolling_mean_24", "rolling_std_24"]).reset_index(drop=True)

print(f"Filas antes de dropna: {len(df)} — después: {len(df_model)}")
FEATURES_CICLICAS = [
    "lag_1", "lag_24", "lag_168",
    "rolling_mean_24", "rolling_std_24",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "holiday",
    "temp_exog", "hum_exog", "windspeed_exog",
]
TARGET = "cnt"
df_model[["datetime"] + FEATURES_CICLICAS + [TARGET]].head()

### Split temporal (NO aleatorio)

En series de tiempo el split de train/test **siempre** debe respetar el
orden cronológico: entrenamos con el pasado y validamos con el futuro. Usamos
el primer 80% de las horas (en orden) para entrenar y el último 20% para
probar.

In [ ]:
n = len(df_model)
corte = int(n * 0.8)

train = df_model.iloc[:corte].copy()
test = df_model.iloc[corte:].copy()

print(f"Train: {len(train)} filas, {train['datetime'].min()} → {train['datetime'].max()}")
print(f"Test:  {len(test)} filas, {test['datetime'].min()} → {test['datetime'].max()}")

X_train, y_train = train[FEATURES_CICLICAS], train[TARGET]
X_test, y_test = test[FEATURES_CICLICAS], test[TARGET]

## 8. Modelos de ML para series de tiempo reformuladas

Una vez tenemos $X$ / $y$, cualquier modelo supervisado aplica. Comparamos
tres:

1. **Baseline naive**: predecir el valor de hace 24 horas
   ($\hat{y}_t = y_{t-24}$, es decir, "asumir que hoy se parece a ayer a esta
   misma hora"). Todo modelo de ML **debe** superar este baseline para
   justificar su complejidad.
2. **Regresión lineal**: el modelo más simple; también podríamos usar
   Ridge/Lasso si quisiéramos regularización (útil si tuviéramos muchas más
   variables correlacionadas entre sí).
3. **Gradient Boosting** (`HistGradientBoostingRegressor` de scikit-learn):
   captura interacciones no lineales entre features (ej. "hora pico" solo
   importa en día laboral) sin que tengamos que especificarlas a mano.

Random Forest es otra alternativa de árboles (mencionada en la teoría); no la
entrenamos aparte aquí para no repetir, pero el código es exactamente
análogo (`sklearn.ensemble.RandomForestRegressor`).

Para problemas más complejos (series únicas muy largas, múltiples
estacionalidades, relaciones no lineales complejas en el tiempo) existen
alternativas que **no** implementamos en este curso por restricciones del
entorno, pero que debes conocer:

- **RNN / LSTM** (redes neuronales recurrentes): aprenden representaciones
  temporales directamente de secuencias, sin necesidad de definir lags a mano.
  Requieren mucho más dato y cómputo (TensorFlow/PyTorch).
- **Prophet** (Meta/Facebook): modelo aditivo con tendencia + estacionalidad +
  festivos, pensado para ser robusto y fácil de usar con poca afinación,
  popular en pronóstico de negocio.

In [ ]:
# --- 1) Baseline naive: predicción = valor de hace 24 horas ---
y_pred_naive = test["lag_24"].values

# --- 2) Regresión lineal ---
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train, y_train)
y_pred_lineal = modelo_lineal.predict(X_test)

# --- 3) Gradient Boosting ---
modelo_hgb = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
modelo_hgb.fit(X_train, y_train)
y_pred_hgb = modelo_hgb.predict(X_test)

print("Modelos entrenados.")

## 9. Métrica WMAPE

El **MAPE** (Mean Absolute Percentage Error) es muy usado en pronóstico de
demanda, pero tiene un problema grave: si $y_t \approx 0$ para algún $t$, el
error porcentual explota (división por un número cercano a cero), y puede
dominar completamente el promedio aunque el modelo esté prediciendo bien en
el resto de la serie.

El **WMAPE** (Weighted Mean Absolute Percentage Error) resuelve esto
ponderando el error por la magnitud real, en lugar de promediar errores
porcentuales individuales:

$$\text{WMAPE} = \frac{\sum_t |y_t - \hat{y}_t|}{\sum_t |y_t|}$$

Ventajas:

- Es robusto a observaciones cercanas a cero (no hay división individual por
  valores pequeños).
- Es directamente interpretable como "el error total equivale a un X% de la
  demanda total".
- Es el estándar de facto en la industria de planeación de demanda (retail,
  supply chain, manufactura).

No está implementada en scikit-learn, así que la escribimos manualmente
(junto con RMSE, que sí está en sklearn pero la dejamos explícita por
claridad):

In [ ]:
def wmape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


resultados = pd.DataFrame(
    [
        {"modelo": "Naive (lag 24h)", "WMAPE": wmape(y_test, y_pred_naive), "RMSE": rmse(y_test, y_pred_naive)},
        {"modelo": "Regresión lineal", "WMAPE": wmape(y_test, y_pred_lineal), "RMSE": rmse(y_test, y_pred_lineal)},
        {"modelo": "HistGradientBoosting", "WMAPE": wmape(y_test, y_pred_hgb), "RMSE": rmse(y_test, y_pred_hgb)},
    ]
).set_index("modelo")

resultados

In [ ]:
fig, ax = plt.subplots()
colores = [COLOR_NAIVE, COLOR_LINEAL, COLOR_HGB]
ax.bar(resultados.index, resultados["WMAPE"], color=colores)
ax.set_ylabel("WMAPE (menor es mejor)")
ax.set_title("Comparación de modelos — Bike Sharing (test = último 20% cronológico)")
for i, v in enumerate(resultados["WMAPE"]):
    ax.text(i, v + 0.005, f"{v:.1%}", ha="center", va="bottom")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
muestra_test = test.iloc[: 24 * 5]  # primeras 5 días del test
idx = muestra_test.index
ax.plot(muestra_test["datetime"], muestra_test["cnt"], color=COLOR_SERIE, label="Real", linewidth=1.5)
ax.plot(muestra_test["datetime"], pd.Series(y_pred_hgb, index=test.index).loc[idx], color=COLOR_HGB, label="HistGradientBoosting", linewidth=1.2, linestyle="--")
ax.plot(muestra_test["datetime"], pd.Series(y_pred_naive, index=test.index).loc[idx], color=COLOR_NAIVE, label="Naive (lag 24h)", linewidth=1, linestyle=":")
ax.set_title("Predicción vs. real — primeros 5 días del test")
ax.set_xlabel("Fecha")
ax.set_ylabel("Alquileres por hora")
ax.legend()
fig.tight_layout()
plt.show()

El modelo de Gradient Boosting debería reducir el WMAPE frente al
baseline naive y frente a la regresión lineal, mostrando el valor de: (a)
combinar varios lags con calendario y rolling, y (b) usar un modelo capaz de
capturar interacciones no lineales entre esas variables.

## 10. Por qué K-Fold aleatorio NO sirve en series de tiempo

La validación cruzada **K-Fold clásica** mezcla aleatoriamente las filas y
entrena/valida en distintas combinaciones. En series de tiempo esto es un
error grave: si una fila del "futuro" (ej. la semana 30) termina en el fold de
**entrenamiento** y una fila del "pasado" (ej. semana 10) termina en el fold
de **validación**, el modelo está usando información del futuro para predecir
el pasado — esto es **fuga de información (data leakage)** y hace que el
error de validación **subestime** dramáticamente el error real que tendría el
modelo en producción (donde nunca tenemos acceso al futuro).

### Alternativas correctas

**Walk-Forward Validation (ventana expansiva)**: cada fold entrena con
**todos** los datos disponibles hasta un punto en el tiempo, y valida en el
bloque inmediatamente siguiente. El conjunto de entrenamiento **crece** en
cada fold. Es la alternativa por defecto y la que implementamos con
`sklearn.model_selection.TimeSeriesSplit`.

**Validación con ventana deslizante (sliding window)**: el tamaño del
conjunto de entrenamiento se mantiene **fijo** y se desplaza hacia adelante en
el tiempo (en vez de crecer, "desliza"). Es útil cuando sospechamos **concept
drift** (el proceso generador cambia con el tiempo) y no queremos que datos
muy antiguos, ya no representativos, sigan influyendo el modelo indefinidamente.

Implementamos Walk-Forward con `TimeSeriesSplit`, usando `df_model` completo
(ordenado cronológicamente) y HistGradientBoosting:

In [ ]:
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

X_full = df_model[FEATURES_CICLICAS]
y_full = df_model[TARGET]

resultados_folds = []
for i, (idx_train, idx_val) in enumerate(tscv.split(X_full), start=1):
    modelo_fold = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    modelo_fold.fit(X_full.iloc[idx_train], y_full.iloc[idx_train])
    y_pred_fold = modelo_fold.predict(X_full.iloc[idx_val])

    resultados_folds.append(
        {
            "fold": i,
            "n_train": len(idx_train),
            "n_val": len(idx_val),
            "rango_val": f"{df_model['datetime'].iloc[idx_val[0]].date()} → {df_model['datetime'].iloc[idx_val[-1]].date()}",
            "WMAPE": wmape(y_full.iloc[idx_val], y_pred_fold),
        }
    )

resultados_folds = pd.DataFrame(resultados_folds).set_index("fold")
resultados_folds

In [ ]:
fig, ax = plt.subplots()
ax.plot(resultados_folds.index, resultados_folds["WMAPE"], marker="o", color=COLOR_HGB, linewidth=2)
ax.set_xlabel("Fold (walk-forward, ventana expansiva)")
ax.set_ylabel("WMAPE")
ax.set_title("Walk-Forward Validation: WMAPE por fold (TimeSeriesSplit)")
ax.set_xticks(resultados_folds.index)
fig.tight_layout()
plt.show()

print(f"WMAPE promedio walk-forward: {resultados_folds['WMAPE'].mean():.2%}")
print(f"WMAPE desviación estándar entre folds: {resultados_folds['WMAPE'].std():.2%}")

Observa que el WMAPE **varía entre folds** — esto es información
valiosa: nos dice qué tan estable es el desempeño del modelo en distintos
periodos, y puede alertarnos sobre estacionalidades anuales, cambios de
comportamiento, o simplemente periodos más difíciles de predecir (ej. cambios
de estación climática). Un solo número de un único split train/test esconde
esta variabilidad.

## Resumen de conceptos clave

**Desbalance de clases**

- La **accuracy engaña** cuando la clase positiva es rara: un modelo que
  siempre predice la clase mayoritaria puede tener accuracy altísima y ser
  inútil (recall = 0 sobre la clase de interés).
- Hay que evaluar con **precision, recall, F1 y PR-AUC**, y se puede
  mitigar el desbalance con **undersampling, oversampling, Tomek Links o
  NearMiss**.

**Series de tiempo**

- Una serie de tiempo tiene observaciones **ordenadas y dependientes**: no
  se puede mezclar aleatoriamente ni asumir independencia (i.i.d.) entre
  filas.
- El enfoque práctico central de la sesión: **reformular la serie como
  problema supervisado** con **lags**, **variables de calendario**
  (incluida la **codificación cíclica seno/coseno**) y **variables
  rolling** — con esa reformulación, cualquier modelo de ML aplica.
- **WMAPE** es la métrica estándar en pronóstico de demanda.
- **K-Fold aleatorio filtra información del futuro** en series de tiempo
  (data leakage) y subestima el error real. La alternativa correcta es
  **Walk-Forward Validation** (`TimeSeriesSplit`, ventana expansiva).

**El hilo común de hoy:** en ambos problemas, la validación "por
defecto" (accuracy simple; K-Fold aleatorio) da una lectura optimista y
engañosa del desempeño real del modelo — y en ambos casos existe una
alternativa específica (PR-AUC + balanceo; Walk-Forward Validation) que sí
refleja el problema real.

### Próxima sesión

En la Sesión 4 veremos **sistemas de recomendación**: cómo completar una
matriz de utilidad muy sparse con filtrado basado en contenido y filtrado
colaborativo (factorización de matrices). Reaparece el corte temporal que
vimos hoy —allí es *por usuario*— y con él la misma advertencia sobre las
particiones aleatorias.

---

# Ejercicio

La parte 1 se resuelve sobre el dataset de fraude retomado al inicio
(`X_train_fraude`, `y_train_fraude`, `rf_under`). Las partes 2-3 se
resuelven sobre lo construido en la sección de series de tiempo: `df`,
`df_model`, `train`, `test`, `FEATURES_CICLICAS` y las funciones `wmape` /
`rmse`.

| # | Parte | Tiempo sugerido |
|---|---|---|
| 1 | Undersampling informado (Tomek Links) vs. aleatorio | 10 min |
| 2 | Demostrar el data leakage: walk-forward vs. partición aleatoria | 10 min |
| 3 | Codificación cíclica vs. codificación categórica simple de la hora | 10 min |

Si una parte se atasca, pasen a la siguiente: valen más las tres intentadas que una perfecta.

### Ejercicio 1 — Undersampling informado (Tomek Links) vs. aleatorio

Implementa una versión simple de **Tomek Links** sobre `X_train_fraude`/`y_train_fraude`:
para cada observación de la clase minoritaria, encuentra su vecino más
cercano (usa `NearestNeighbors` de scikit-learn sobre variables
estandarizadas); si ese vecino más cercano pertenece a la clase mayoritaria
**y** la observación minoritaria es también el vecino más cercano de ese
vecino (es decir, son mutuamente el vecino más cercano el uno del otro),
entonces esa pareja forma un Tomek Link: elimina la observación
**mayoritaria** del link. Entrena un Random Forest sobre el training set
resultante y compara sus métricas contra el undersampling aleatorio de la
sección 4.4 (F1, recall, PR-AUC).

In [ ]:
# TODO: estandariza las variables (ej. StandardScaler) para calcular distancias
# TODO: usa NearestNeighbors para encontrar, para cada punto, su vecino más cercano
# TODO: identifica los Tomek Links (pares mutuamente más cercanos de clases opuestas)
# TODO: elimina las observaciones mayoritarias que forman parte de un Tomek Link
# TODO: entrena un RandomForestClassifier sobre ese training set y compara métricas

### Ejercicio 2 — Demostrar el data leakage: walk-forward vs. partición aleatoria

Compara el WMAPE promedio obtenido con **walk-forward validation**
(`TimeSeriesSplit`, ya calculado arriba en `resultados_folds`) contra el
WMAPE promedio obtenido con una **partición aleatoria ingenua** tipo K-Fold
(`sklearn.model_selection.KFold` con `shuffle=True`) usando el **mismo**
modelo y las **mismas** features. La hipótesis a comprobar: el K-Fold
aleatorio va a mostrar un error **optimistamente bajo** porque usa datos del
futuro para "adivinar" el pasado.

In [ ]:
# TODO: Ejercicio 2
# 1. Importa KFold de sklearn.model_selection.
# 2. Usa KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE) sobre X_full / y_full.
# 3. Para cada fold, entrena HistGradientBoostingRegressor y calcula wmape() en el fold de validación.
# 4. Compara el WMAPE promedio contra resultados_folds["WMAPE"].mean() (walk-forward).

### Ejercicio 3 — Codificación cíclica vs. codificación categórica simple de la hora

Compara el desempeño de la **regresión lineal** cuando la hora se codifica de
dos formas distintas:

- **Cíclica** (la que usamos en el modelo principal): `hour_sin`, `hour_cos`.
- **Simple/ordinal**: la hora como un solo número entero `hr` (0-23), sin
  transformar.

Entrena una regresión lineal con cada codificación (dejando las demás
features iguales) y compara el WMAPE en el mismo conjunto de test. ¿Cuál
esperarías que funcione mejor para un modelo **lineal**, y por qué?

In [ ]:
# TODO: Ejercicio 3
# 1. Crea FEATURES_HORA_CICLICA = FEATURES_CICLICAS (ya usa hour_sin/hour_cos).
# 2. Crea FEATURES_HORA_SIMPLE = mismas features pero reemplazando hour_sin/hour_cos por "hr" (entero).
# 3. Entrena LinearRegression con cada set sobre train/test (los mismos que usamos arriba).
# 4. Calcula wmape() para cada uno y compara.

---

### Ejercicio extra (opcional, sin cronometrar) — Más lags y variables rolling

Agrega al menos dos variables nuevas al set de features: por ejemplo
`lag_2`, `lag_48` (hace 2 días) y/o `rolling_mean_168` (media móvil de la
última semana). Reentrena `HistGradientBoostingRegressor` con el nuevo set de
features y compara el WMAPE en test contra el modelo original (`resultados`).
¿Mejora, empeora, o el cambio es marginal? ¿Por qué crees que pasa eso?

In [ ]:
# TODO: Ejercicio 1
# 1. Crea columnas nuevas en `df` (ej. df["lag_2"] = df["cnt"].shift(2), etc.)
# 2. Vuelve a hacer dropna() y el split temporal 80/20.
# 3. Entrena un nuevo HistGradientBoostingRegressor con el set de features ampliado.
# 4. Calcula wmape() en test y compara contra resultados.loc["HistGradientBoosting", "WMAPE"].